In [1]:
import pickle

with open("X_train_tfidf.pkl", "rb") as f:
    X_train = pickle.load(f)

with open("X_test_tfidf.pkl", "rb") as f:
    X_test = pickle.load(f)

with open("y_train_tfidf.pkl", "rb") as f:
    y_train = pickle.load(f)

with open("y_test_tfidf.pkl", "rb") as f:
    y_test = pickle.load(f)

with open("tfidf.pkl", "rb") as f:
    tfidf = pickle.load(f)

In [5]:
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

pipelines = {
    "LR + SVD": Pipeline([
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("clf", LogisticRegression(C=1, max_iter=2000))
    ]),

    "SVM Linear + SVD": Pipeline([
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("clf", SVC(kernel="linear", C=1))
    ])
}

In [6]:
for name, model in pipelines.items():
    print("Training:", name)

    model.fit(X_train, y_train)   # X_train is already TF-IDF

    preds = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1:", f1_score(y_test, preds))

Training: LR + SVD
Accuracy: 0.7573282075925036
F1: 0.7513540128015755
Training: SVM Linear + SVD
Accuracy: 0.758769822200865
F1: 0.749


In [7]:
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

print("MODEL SELECTION WITH PRECOMPUTED TF-IDF + SVD")

# -----------------------------
# STEP 1: CV setup
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# -----------------------------
# STEP 2: Pipelines (NO TF-IDF HERE)
# -----------------------------
models = {
    "LR + SVD": Pipeline([
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("clf", LogisticRegression(C=1, max_iter=2000))
    ]),

    "SVM Linear + SVD": Pipeline([
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("clf", SVC(kernel="linear", C=1))
    ])
}

# -----------------------------
# STEP 3: Cross-validation
# -----------------------------
results = {}

for name, model in models.items():
    print(f"\nEvaluating {name}...")

    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")

    results[name] = {
        "CV Mean": scores.mean(),
        "CV Std": scores.std()
    }

    print("CV Mean:", scores.mean())
    print("CV Std:", scores.std())

# -----------------------------
# STEP 4: Ranking
# -----------------------------
df_results = pd.DataFrame(results).T
df_results = df_results.sort_values(by="CV Mean", ascending=False)

print("\nMODEL RANKING")
print(df_results)

best_model_name = df_results.index[0]
best_model = models[best_model_name]

print("\nSELECTED BEST MODEL:", best_model_name)

# -----------------------------
# STEP 5: Train best model
# -----------------------------
best_model.fit(X_train, y_train)

# -----------------------------
# STEP 6: Evaluation
# -----------------------------
print("\nFINAL TEST RESULTS")

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("\nTRAIN METRICS")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("F1:", f1_score(y_train, y_train_pred))

print("\nTEST METRICS")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1:", f1_score(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

# -----------------------------
# STEP 7: Overfitting check
# -----------------------------
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

gap = train_acc - test_acc

print("\nOVERFITTING ANALYSIS")
print("Train-Test Gap:", gap)

if gap < 0.03:
    print("No significant overfitting")
elif gap < 0.07:
    print("Mild overfitting")
else:
    print("Strong overfitting still present")

MODEL SELECTION WITH PRECOMPUTED TF-IDF + SVD

Evaluating LR + SVD...
CV Mean: 0.7635462866712868
CV Std: 0.008440414607989164

Evaluating SVM Linear + SVD...
CV Mean: 0.76378696003696
CV Std: 0.007588018815616626

MODEL RANKING
                   CV Mean    CV Std
SVM Linear + SVD  0.763787  0.007588
LR + SVD          0.763546  0.008440

SELECTED BEST MODEL: SVM Linear + SVD

FINAL TEST RESULTS

TRAIN METRICS
Accuracy: 0.7857743602066563
F1: 0.7668976336776049

TEST METRICS
Accuracy: 0.758769822200865
F1: 0.749

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.82      0.77      1014
           1       0.80      0.70      0.75      1067

    accuracy                           0.76      2081
   macro avg       0.76      0.76      0.76      2081
weighted avg       0.76      0.76      0.76      2081


Confusion Matrix:
[[830 184]
 [318 749]]

OVERFITTING ANALYSIS
Train-Test Gap: 0.02700453800579128
No significant overfitting
